In [6]:
from google.colab import files
uploaded = files.upload()

Saving talabat_enhanced_orders2.csv to talabat_enhanced_orders2 (1).csv


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [8]:
df = pd.read_csv("talabat_enhanced_orders2.csv")

In [9]:
results = []

top_k_values = [3, 5, 6]

for top_k in top_k_values:

    df_exp = df.copy()
    df_exp["Order_Time"] = pd.to_datetime(df_exp["Order_Time"])
    df_exp["order_hour"] = df_exp["Order_Time"].dt.hour
    df_exp["order_dayofweek"] = df_exp["Order_Time"].dt.dayofweek
    df_exp["is_weekend"] = df_exp["order_dayofweek"].isin([5,6])
    df_exp["price_per_item"] = df_exp["Total_Price"] / df_exp["Quantity"]

    top_items = df_exp["Item_Name"].value_counts().head(top_k).index

    df_exp["Item_Name_reduced"] = df_exp["Item_Name"].apply(
        lambda x: x if x in top_items else "Other"
    )

    df_exp["price_tier"] = pd.cut(
        df_exp["Total_Price"],
        bins=[0,100,250,500,np.inf],
        labels=["low","medium","high","very_high"]
    )

    drop_cols = [
        "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
        "Total_Price", "Order_Time", "Item_Name"
    ]

    X = df_exp.drop(columns=[c for c in drop_cols if c in df_exp.columns])
    y = df_exp["Order_Status"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns
    numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns

    preprocess = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ], remainder="passthrough")

    model = Pipeline([
        ("preprocess", preprocess),
        ("classifier", RandomForestClassifier(random_state=42))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    rf = model.named_steps["classifier"]
    feature_names = model.named_steps["preprocess"].get_feature_names_out()

    importances = pd.Series(rf.feature_importances_, index=feature_names)
    top_features = importances.sort_values(ascending=False).head(5)

    results.append({
        "top_k": top_k,
        "accuracy": acc,
        "top_features": top_features
    })

In [5]:
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Delivery_Time', 'City', 'Payment_Method', 'Order_Status',
       'Driver_Vehicle', 'Traffic_Level', 'Driver_Availability',
       'Item_Name_reduced', 'price_tier'],
      dtype='object'))])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
for r in results:
    print("="*40)
    print(f"top_k = {r['top_k']}")
    print(f"Accuracy = {r['accuracy']:.4f}")
    print("Top Features:")
    print(r["top_features"])

top_k = 3
Accuracy = 1.0000
Top Features:
cat__Order_Status_Delivered     0.307699
cat__Order_Status_Cancelled     0.282744
cat__Order_Status_In Transit    0.217635
remainder__Customer_Lon         0.008022
remainder__Driver_Lon           0.007992
dtype: float64
top_k = 5
Accuracy = 1.0000
Top Features:
cat__Order_Status_Delivered        0.309804
cat__Order_Status_Cancelled        0.297003
cat__Order_Status_In Transit       0.178727
remainder__Delivery_Distance_km    0.008988
remainder__Customer_Lon            0.008847
dtype: float64
top_k = 6
Accuracy = 1.0000
Top Features:
cat__Order_Status_Delivered     0.301541
cat__Order_Status_Cancelled     0.288248
cat__Order_Status_In Transit    0.198868
remainder__price_per_item       0.008720
remainder__Restaurant_Lat       0.008670
dtype: float64
